J24_Projet pratique 2 — Ventes & KPI

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

Heure 1 : Comprendre et nettoyer le dataset

In [52]:
#1
df = pd.read_csv("J24_dataset_ventes.csv")
df.info()
df.head()
df.tail()
#2
df.shape
df.dtypes
df["date"] = pd.to_datetime(df["date"],format="mixed",dayfirst=True)
print(df["date"].min())
print(df["date"].max())
#3
Nomb_man = df.isna().sum()
Tot = len(df)
perc = ((Nomb_man) / (Tot)) * 100
perc.round(2)
#4
df[df.duplicated()]
df = df.drop_duplicates()
#5
df["city"].value_counts(dropna=False)
df["channel"].value_counts(dropna=False)
df["category"].value_counts(dropna=False)
df["customer_segment"].value_counts(dropna=False)

df["sale_id"] = df["sale_id"].str.strip().str.title()
df["customer_id"] = df["customer_id"].str.strip().str.title()
df["city"] = df["city"].str.strip().str.lower()
df["region"] = df["region"].str.strip().str.title()
df["channel"] = df["channel"].str.strip().str.lower()
df["category"] = df["category"].str.strip().str.lower()
df["product"] = df["product"].str.strip().str.title()
#6
corrections_categorie = {
    "Electronic": "Electronique",
    "Burau": "Bureau",  
    "Maisson": "Maison",  
    }

df["category"] = df["category"].replace(corrections_categorie)

corrections_canaux = {
    "Comercial": "Commercial",
    "Commercialle ": "Commercial",  
    "Commerciale": "Commercial",
     "EnLigne": "En Ligne",
         "En ligne": "En Ligne",  
         "Magassin": "Magasin", 
         "Magasine" : "Magasin" 
    }
df["channel"] = df["channel"].replace(corrections_canaux)

corrections_ville = {
    "Casa": "Casablanca",
    "casa": "Casablanca",  
    "Rbat": "Rabat",
     "Raba": "Rabat",
      "Tangier": "Tanger",  
     "Marrakach": "Marrakech", 
    "Marakech" : "Marrakech",
    "Marrakesh" : "Marrakech",
    "Fès" : "Fes"
    }
df["city"] = df["city"].replace(corrections_ville)

df["customer_segment"] = (
    df["customer_segment"]
    .str.strip()
    .str.lower()
    .replace({
        "particulier": "Particulier",
        "pme": "PME",
        "grand compte": "Grand compte"
    })
)
#7
df["date"] = pd.to_datetime(df["date"], format="mixed")
#8
df[
    [
        "quantity",
        "discount_pct",
        "unit_price_mad",
        "unit_cost_mad"
    ]].describe()
""" sns.scatterplot(data = df, x = "quantity",y="unit_price_mad", hue = "product")
sns.scatterplot(data = df, x = "quantity",y="unit_cost_mad", hue = "product")
sns.scatterplot(data = df, x = "quantity",y="discount_pct", hue = "product") """
#9
df["customer_segment"] = df["customer_segment"].fillna("Inconnu") #Variable categorielle
df["city"] = df["city"].fillna(df["city"].mode()[0]) #Variable categorielle
df["discount_pct"] = df["discount_pct"].fillna(df["discount_pct"].median()) #Variable numerique
df["salesperson"] = df["salesperson"].fillna("Inconnu") #Variable categorielle
#10
df["revenue_mad"] = df["quantity"] * df["unit_price_mad"] * (1-(df["discount_pct"] / 100))
df["total_cost_mad"] = df["quantity"] * df["unit_cost_mad"]
df["profit_mad"] = df["revenue_mad"] - df["total_cost_mad"]
df["margin_pct"] = (df["profit_mad"] / df["revenue_mad"]) * 100

df.info()
df.dtypes
df.describe()
df.isna().sum().sum()
df.duplicated().sum()

<class 'pandas.DataFrame'>
RangeIndex: 906 entries, 0 to 905
Data columns (total 15 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   sale_id           906 non-null    str    
 1   date              906 non-null    str    
 2   customer_id       906 non-null    str    
 3   customer_segment  898 non-null    str    
 4   city              896 non-null    str    
 5   region            906 non-null    str    
 6   channel           906 non-null    str    
 7   category          906 non-null    str    
 8   product           906 non-null    str    
 9   quantity          906 non-null    int64  
 10  unit_price_mad    906 non-null    float64
 11  unit_cost_mad     906 non-null    float64
 12  discount_pct      894 non-null    float64
 13  salesperson       900 non-null    str    
 14  payment_method    906 non-null    str    
dtypes: float64(3), int64(1), str(11)
memory usage: 106.3 KB
2026-01-01 00:00:00
2026-12-08 00:00:00
<class

np.int64(0)

Heure 2 :  GroupBy, Pivot Table, KPI et segmentation

In [ ]:
#1
CA_Total = df["revenue_mad"].sum()
CA_Total.round(2)
Profit_tot = df["profit_mad"].sum()
Profit_tot.round(2)
quantite_tot_ven = df["quantity"].sum()
quantite_tot_ven.round(2)
Valeur_moy_tr = (df["revenue_mad"]).mean().round(2)
Valeur_moy_tr
#2
df_sorted = df.sort_values(by="revenue_mad", ascending=False)["product"]
df_sorted.head(5)
#3
df.groupby("category").agg(nombre_transactions=("sale_id","count"),
quantite_vendue = ("quantity","sum"),
chiffr_aff = ("revenue_mad","sum"),    
profit_moy = ("profit_mad","mean"),    
marge_moy = ("margin_pct","mean")).round(2)
#4
df.groupby("region").agg(chiffr_aff = ("revenue_mad","sum"),    
profit_sum = ("profit_mad","sum"),
transactions_moy=("revenue_mad", "mean")
).round(2)
#5
df.groupby("channel").agg(chiffr_aff = ("revenue_mad","sum"),    
profit_sum = ("profit_mad","sum"),
marge_moy = ("margin_pct","mean"),
nombre_transactions=("sale_id","count")
).round(2)
#6
df["month"] = df["date"].dt.month
"""sns.lineplot(data=df, x= "month" , y = "revenue_mad", marker = "o")
plt.title("Evolution du CA")

sns.lineplot(data=df, x= "month" , y = "profit_mad", marker = "o")
plt.title("Evolution du profit") """
#7
pd.pivot_table(df,values="revenue_mad",index = "category", columns = "channel",aggfunc="sum", )
#8
df.groupby(["customer_segment"]).agg(effectif_transactions=("sale_id","count"),
chiffr_aff = ("revenue_mad","sum"),    
panier_moyen = ("revenue_mad","mean"),
profit_moy= ("profit_mad","mean"),
marge_moy = ("margin_pct","mean")).round(2)
#9
sales_by_person = (
    df.groupby("salesperson")
    .agg(
        nb_transactions=("sale_id", "count"),
        total_ca=("revenue_mad", "sum"),
        total_profit=("profit_mad", "sum"),).reset_index())
plus_20 = sales_by_person[sales_by_person["nb_transactions"] >= 20]
trois_meill_CA = plus_20.sort_values(by = "total_ca",ascending=False).head(3)
trois_meill_profit = plus_20.sort_values(by = "total_profit",ascending=False).head(3)
print(trois_meill_CA)
print(trois_meill_profit)

     salesperson  nb_transactions      total_ca  total_profit
2  Commercial_03               99  1.048989e+06   278315.9455
0  Commercial_01              100  8.929619e+05   260734.2120
6  Commercial_07              108  8.158306e+05   213554.1580
     salesperson  nb_transactions      total_ca  total_profit
2  Commercial_03               99  1.048989e+06   278315.9455
0  Commercial_01              100  8.929619e+05   260734.2120
7  Commercial_08              101  7.092428e+05   228211.5470
